In [11]:
import pandas as pd
import polars as pl
import numpy as np
import time
from pathlib import Path

# --- FONCTION UTILITAIRE DE BENCHMARK ---
def benchmark(name,func):
    start = time.time()
    func()
    return time.time() - start

def print_results(title, results):
    print(f"\n=== {title} ===")
    for k, (pandas_time, polars_time) in results.items():
        print(f"{k:35} | pandas: {pandas_time:.4f} s | polars: {polars_time:.4f} s")

# --- DATASET ---
n = 1_000_000
date_range = pd.date_range("2024-01-01", periods=n, freq="min")
categories = ["a", "b", "c"]

df_pd = pd.DataFrame({
    "timestamp":  np.tile(date_range, len(categories)),
    "category": np.repeat(categories, len(date_range)),
    "value": np.random.randn(n * len(categories)),
})

df_pl = pl.from_pandas(df_pd)

# --- OPÉRATIONS SUR TIME SERIES ---
ts_results = {}

ts_results["Filter (month == 2)"] = (
    benchmark("pandas", lambda: df_pd[df_pd["timestamp"].dt.month == 2]),
    benchmark("polars", lambda: df_pl.filter(pl.col("timestamp").dt.month() == 2))
)

ts_results["Filter (category == a)"] = (
    benchmark("pandas", lambda: df_pd[df_pd["category"] == "a"]),
    benchmark("polars", lambda: df_pl.filter(pl.col("category") == 'a'))
)

ts_results["Mean per day"] = (
    benchmark("pandas", lambda: df_pd.set_index(["timestamp"]).resample("D")["value"].mean()),
    benchmark("polars", lambda: df_pl.sort('timestamp', 'category').group_by_dynamic("timestamp", every="1d", period="1d").agg([
        pl.col("value").mean().alias("mean_value")
    ]))
)


ts_results["Extract time features"] = (
    benchmark("pandas", lambda: df_pd["timestamp"].dt.hour + df_pd["timestamp"].dt.dayofweek),
    benchmark("polars", lambda: df_pl.with_columns([
        pl.col("timestamp").dt.hour().alias('hour'),
        pl.col("timestamp").dt.weekday().alias('weekday')
    ]))
)



# --- I/O CSV & PARQUET ---
io_results = {}
output_dir = Path("benchmark_io")
output_dir.mkdir(exist_ok=True)
csv_path = output_dir / "data.csv"
parquet_path = output_dir / "data.parquet"

# Sous-échantillonnage pour I/O
df_pd_small = df_pd.iloc[:100_000]
df_pl_small = pl.from_pandas(df_pd_small)

# CSV
io_results["CSV write"] = (
    benchmark("pandas", lambda: df_pd_small.to_csv(csv_path, index=False)),
    benchmark("polars", lambda: df_pl_small.write_csv(csv_path))
)

io_results["CSV read"] = (
    benchmark("pandas", lambda: pd.read_csv(csv_path, parse_dates=["timestamp"])),
    benchmark("polars", lambda: pl.read_csv(csv_path, try_parse_dates=True))
)

# Parquet
io_results["Parquet write"] = (
    benchmark("pandas", lambda: df_pd_small.to_parquet(parquet_path, index=False, engine="pyarrow")),
    benchmark("polars", lambda: df_pl_small.write_parquet(parquet_path))
)

io_results["Parquet read"] = (
    benchmark("pandas", lambda: pd.read_parquet(parquet_path, engine="pyarrow")),
    benchmark("polars", lambda: pl.read_parquet(parquet_path))
)

# --- AFFICHAGE DES RÉSULTATS ---
print_results("Benchmarks sur les opérations TimeSeries", ts_results)
print_results("Benchmarks I/O CSV & Parquet", io_results)

## -- Pivot COMPARISON -- 

# # Wide-format DataFrames for pivot benchmarks
df_pandas_wide = df_pd.pivot_table(
    index="timestamp", columns="category", values="value",
).reset_index()
df_polars_wide = df_pl.pivot(
    index="timestamp", on="category", values="value",
)

pivot_results = {}
pivot_results["Melt"] = (
    benchmark("pandas", lambda: df_pd.melt(id_vars=["timestamp", "category"], value_vars=["value"])),
    benchmark("polars", lambda: df_pl.melt(id_vars=["timestamp", "category"], value_vars=["value"]))
)


pivot_results['Pivot'] = (
        benchmark("pandas", lambda: df_pandas_wide.pivot(index="timestamp", columns="category", values=["a", "b", "c"])),
        benchmark("polars", lambda: df_polars_wide.pivot(index="timestamp", columns="category", values=["a", "b", "c"]))
)

print_results("Benchmarks on pivot and melting", pivot_results)



=== Benchmarks sur les opérations TimeSeries ===
Filter (month == 2)                 | pandas: 0.0524 s | polars: 0.0595 s
Filter (category == a)              | pandas: 0.1969 s | polars: 0.0091 s
Mean per day                        | pandas: 0.4215 s | polars: 0.0991 s
Extract time features               | pandas: 0.1212 s | polars: 0.0585 s

=== Benchmarks I/O CSV & Parquet ===
CSV write                           | pandas: 0.3811 s | polars: 0.0303 s
CSV read                            | pandas: 0.0907 s | polars: 0.0087 s
Parquet write                       | pandas: 0.0380 s | polars: 0.0112 s
Parquet read                        | pandas: 0.0095 s | polars: 0.0099 s


ValueError: value_name (value) cannot match an element in the DataFrame columns.

|  Opération                                 |  `pandas`                           |  `polars`                                                   |
| -------------------------------------------- | ------------------------------------- | ------------------------------------------------------------- |
|  Créer un DataFrame                         | `pd.DataFrame({...})`                 | `pl.DataFrame({...})`                                         |
|  Voir les premières lignes                 | `df.head()`                           | `df.head()`                                                   |
|  Accéder à une colonne                     | `df["col"]` ou `df.col`               | `df["col"]` ou `df.select("col")` ou `df["col"]`              |
|  Créer une nouvelle colonne                | `df["z"] = df["x"] + df["y"]`         | `df.with_columns((pl.col("x") + pl.col("y")).alias("z"))`     |
|  Supprimer une colonne                      | `df.drop(columns=["x"])`              | `df.drop("x")`                                                |
|  Supprimer les colonnes d'un type          | `df = df.drop(columns=df.select_dtypes(include='object').columns)` | `df.drop(pl.selectors.bool())` | 
|  Sélection de colonnes                     | `df[["a", "b"]]`                      | `df.select(["a", "b"])`                                       |
|  Sélection par pattern                     | `df.filter(like="foo")`               | `df.select(pl.col("^foo.*$"))` (regex)                        |
|  Filtrage conditionnel                     | `df[df["x"] > 3]`                     | `df.filter(pl.col("x") > 3)`                                  |
|  Filtrage avec plusieurs conditions        | `df[(df["x"] > 3) & (df["y"] < 1)]`   | `df.filter((pl.col("x") > 3) & (pl.col("y") < 1))`            |
|  Colonnes numériques uniquement            | `df.select_dtypes(include=np.number)` | `df.select(pl.selectors.numeric())`                        |
|  Renommer les colonnes                    | `df.rename(columns={"a": "b"})`       | `df.rename({"a": "b"})`                                       |
|  Convertir en datetime                     | `pd.to_datetime(df["ts"])`            | `df_with_columns(pl.col("ts").str.strptime(pl.Datetime)`                      |
|  Extraire l’année/mois/jour d’une datetime | `df["ts"].dt.year`                    | `df_with_columns(pl.col("ts").dt.year())`                                      |
|  Groupby + agg                             | `df.groupby("a").agg({"b": "mean"})`  | `df.groupby("a").agg(pl.col("b").mean())`                     |
|  Rolling mean                              | `df["x"].rolling(window=3).mean()`    | `df_with_columns(pl.col("x").rolling_mean(3))`                                 |
|  Trier                                     | `df.sort_values("x")`                 | `df.sort("x")`                                                |
|  Valeurs uniques                           | `df["x"].unique()`                    | `df.select("x").unique()`                                     |
|  Valeurs nulles                             | `df.isnull()`                         | `df.select(pl.all().is_null())`                               |
|  Drop NA                                    | `df.dropna()`                         | `df.drop_nulls()`                                             |
|  Cumulative sum                            | `df["x"].cumsum()`                    | `df.select(pl.col("x").cumsum())`                             |
|  Stack / Melt                              | `df.melt()`                           | `df.melt(id_vars=..., value_vars=...)`                        |
|  Pipeline (chained ops)                    | pas naturel                           | `.with_columns(...).filter(...).select(...)`                  |
|  Lazy DataFrame                            | (non dispo)                         | `df.lazy()`                                                   |


In [11]:
df = (
    pl.read_parquet(parquet_path)
    .with_columns([
        (pl.col("value")**2).alias("squared"),
        pl.col("timestamp").dt.hour().alias("hour")
    ])
    .filter(pl.col("squared") > 0)
    .group_by("hour")
    .agg(pl.col("squared").mean().alias("avg_squared"))
    .sort("hour")
)

df


hour,avg_squared
i8,f64
0,0.98668
1,0.967491
2,1.014947
3,0.953232
4,0.972124
…,…
19,0.99142
20,0.945809
21,0.945816


In [15]:
import pandas as pd

df = pd.read_parquet(parquet_path)
# Créer les colonnes "sum" et "hour"
df["squared"] = df["value"] ** 2
df["hour"] = df["timestamp"].dt.hour

# Filtrer les lignes
df_filtered = df[df["squared"] > 0]

# Grouper par heure et faire la moyenne
df_grouped = df_filtered.groupby("hour", as_index=False)["squared"].mean()
df_grouped.rename(columns={"squared": "avg_sum"}, inplace=True)

# Trier par heure
df_grouped = df_grouped.sort_values("hour").reset_index(drop=True)

df_grouped

,hour,avg_sum
0,0,0.986680
1,1,0.967491
2,2,1.014947
3,3,0.953232
4,4,0.972124
5,5,0.988269
6,6,1.001218
7,7,0.997807
8,8,1.036747
9,9,0.986453


In [5]:
df_pd.melt(id_vars=["timestamp", "category"], value_vars=["value"])

ValueError: value_name (value) cannot match an element in the DataFrame columns.